# Agent365 Sampling V2 Interactive Runbook



This notebook documents and reproduces the expected-label-only Sampling V2 workflow. It exercises the three production prototypes:



1. **Random** from `random_sampling`

2. **MinHash LSH 32x4** from `minhash_sampling`

3. **Full-session embedding** from `trace_sampling`



The default mode is artifact-first and does **not** rerun the full experiment, call an LLM judge, post snapshots, or modify Azure resources. Expected result labels are kept separate from selection and are used only for scoring after selected IDs are fixed.

## 1. Configure Paths, Seeds, and Experiment Parameters



Configure repository-relative inputs, deterministic seeds, V2 budgets, and safe execution switches. `FULL_RUN=False` validates the retained artifacts. Set it to `True` only when intentionally creating a new run directory.

In [ ]:
from pathlib import Path

from datetime import datetime, timezone

import hashlib

import importlib.metadata

import json

import platform

import sys



import matplotlib.pyplot as plt

import numpy as np

import pandas as pd



REPO_ROOT = Path.cwd().resolve()

if not (REPO_ROOT / 'random_sampling').is_dir():

    raise RuntimeError('Run this notebook from the singlenotebooks repository root.')



SYNTHETIC_ROOT = REPO_ROOT / 'synthetic_data'

HISTORICAL_PATH = SYNTHETIC_ROOT / 'a365_historical_300' / 'synthetic_observability.a365-otel.json'

DENSE_PATH = SYNTHETIC_ROOT / 'a365_dense_2500' / 'corpus' / 'a365.synthetic.strict.otlp.json'

REFERENCE_DIR = REPO_ROOT / 'outputs_sampling_v2' / 'v2'

REFERENCE_REPORT = REFERENCE_DIR / 'agent365-sampling-v2-report.html'



FULL_RUN = False

RUN_MODE = 'smoke'  # 'smoke' or 'full'; used only when FULL_RUN=True

WRITE_REPORT = False

ENABLE_OPTIONAL_LLM_JUDGE_EXPORT = False

SEED = 13

BUDGET_PCTS = (5, 10, 20, 30, 50)

MINHASH_BANDS = 32

MINHASH_ROWS = 4

MINHASH_PERMUTATIONS = MINHASH_BANDS * MINHASH_ROWS

EMBEDDING_MAX_TOKENS = 8192

RUN_OUTPUT_DIR = REPO_ROOT / 'outputs_sampling_v2' / 'runs' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')



assert RUN_MODE in {'smoke', 'full'}

print({'python': platform.python_version(), 'repo': str(REPO_ROOT), 'full_run': FULL_RUN, 'run_mode': RUN_MODE, 'seed': SEED})

## 2. Validate Required V2 Artifacts and Production Prototypes



Verify the retained reference report, report inputs, synthetic datasets, expected-label loader, session-compression implementation, and the three production prototype packages before doing any work.

In [ ]:
from random_sampling import SamplingEngine, SamplePolicy

from minhash_sampling import BandedMinHashLSHIndex, MinHashConfig

from trace_sampling import FullSessionEmbeddingPrototype

from trace_sampling.representation import SessionEvidencePacketBuilder

from sampling_comparison.v2_experiment import (

    REPRESENTATION_SEED,

    build_v2_precomputed_runtime,

    load_combined_dataset,

    run_v2_experiment_bundle,

    select_ids_for_method,

    slice_dataset,

    with_permuted_labels,

)

from sampling_comparison.v2_outputs import validate_external_eval_snapshot

from sampling_comparison.v2_report import (

    default_inputs,

    load_v2_artifacts,

    validate_v2_artifacts,

    write_v2_html_report,

)



required = {

    'reference_report': REFERENCE_REPORT,

    'aggregate': REFERENCE_DIR / 'aggregate.json',

    'quadrants': REFERENCE_DIR / 'quadrant.json',

    'throughput': REFERENCE_DIR / 'throughput.json',

    'historical_300': HISTORICAL_PATH,

    'dense_2500': DENSE_PATH,

    'random_prototype': REPO_ROOT / 'random_sampling',

    'minhash_lsh_prototype': REPO_ROOT / 'minhash_sampling',

    'full_session_prototype': REPO_ROOT / 'trace_sampling' / 'full_session_prototype.py',

    'compression': REPO_ROOT / 'trace_sampling' / 'representation.py',

}

validation = pd.DataFrame([

    {'resource': name, 'path': str(path), 'exists': path.exists(), 'bytes': path.stat().st_size if path.is_file() else None}

    for name, path in required.items()

])

display(validation)

missing = validation.loc[~validation['exists'], 'resource'].tolist()

if missing:

    raise FileNotFoundError(f'Missing required V2 resources: {missing}')

print('All required V2 resources and production prototypes are present.')

## 3. Load and Validate Synthetic Session Data



Hash the retained synthetic inputs, load both Agent365 OTLP sources into one 2,800-session frame, inspect source and agent distributions, and detect identifier or timestamp problems. Raw conversation text is intentionally not displayed.

In [ ]:
def sha256_file(path: Path) -> str:

    digest = hashlib.sha256()

    with path.open('rb') as handle:

        for chunk in iter(lambda: handle.read(1024 * 1024), b''):

            digest.update(chunk)

    return digest.hexdigest()



corpus_audit = json.loads((REFERENCE_DIR / 'corpus_audit.json').read_text(encoding='utf-8'))

hash_rows = []

for corpus_id, path in {'historical_300': HISTORICAL_PATH, 'dense_2500': DENSE_PATH}.items():

    recorded = corpus_audit['source_files'][corpus_id]

    actual = sha256_file(path)

    hash_rows.append({

        'corpus_id': corpus_id,

        'path': str(path.relative_to(REPO_ROOT)),

        'bytes': path.stat().st_size,

        'actual_sha256': actual,

        'recorded_sha256': recorded['sha256'],

        'hash_match': actual == recorded['sha256'],

    })

hash_table = pd.DataFrame(hash_rows)

display(hash_table)

assert hash_table['hash_match'].all(), 'Synthetic input hash mismatch.'



data = load_combined_dataset(historical_path=str(HISTORICAL_PATH), dense_path=str(DENSE_PATH))

unit_ids = list(data.unit_ids)

timestamps = [unit.ended_at or unit.started_at for unit in data.units]

audit = {

    'sessions': len(data.units),

    'labels': len(data.labels_by_unit),

    'scoped_agents': len(data.scoped_identities),

    'unique_unit_ids': len(set(unit_ids)),

    'missing_timestamps': sum(ts is None for ts in timestamps),

    'duplicate_unit_ids': len(unit_ids) - len(set(unit_ids)),

}

display(pd.DataFrame([audit]))

assert audit == {'sessions': 2800, 'labels': 2800, 'scoped_agents': 105, 'unique_unit_ids': 2800, 'missing_timestamps': 0, 'duplicate_unit_ids': 0}



source_distribution = pd.Series(data.corpus_id_by_unit).value_counts().rename_axis('corpus_id').reset_index(name='sessions')

display(source_distribution)

## 4. Load Existing Expected-Result Labels



Expected labels are already present in the retained synthetic datasets. Validate complete one-to-one label coverage and show aggregate distributions. This step does not invoke an LLM judge.

In [ ]:
label_rows = pd.DataFrame([

    {'unit_id': uid, 'corpus_id': data.corpus_id_by_unit[uid], 'expected_pass': bool(data.labels_by_unit[uid])}

    for uid in data.unit_ids

])

coverage = {

    'sessions': len(data.unit_ids),

    'labeled_sessions': label_rows['unit_id'].nunique(),

    'duplicate_label_ids': int(label_rows['unit_id'].duplicated().sum()),

    'unmatched_sessions': len(set(data.unit_ids) - set(label_rows['unit_id'])),

}

display(pd.DataFrame([coverage]))

display(label_rows.groupby('corpus_id')['expected_pass'].agg(['count', 'sum', 'mean']).rename(columns={'sum': 'pass_count', 'mean': 'pass_rate'}))

assert coverage['labeled_sessions'] == 2800 and coverage['duplicate_label_ids'] == 0 and coverage['unmatched_sessions'] == 0

print('Expected labels are complete and remain separate from selection inputs.')

## 5. Reconstruct Full Sessions from Trace Events



Use the same bounded canonical representation used by the full-session embedding prototype and judge-evidence path. Show structure and audit metadata only; do not print raw messages, arguments, or tool outputs.

In [ ]:
packet_builder = SessionEvidencePacketBuilder()

sample_trace = data.traces[0]

packet_a = packet_builder.build(sample_trace)

packet_b = packet_builder.build(sample_trace)

assert packet_a.canonical_json == packet_b.canonical_json



sample_unit = data.units[0]

sanitized = {

    'unit_id': sample_unit.unit_id,

    'tenant_id': sample_unit.tenant_id,

    'agent_id': sample_unit.agent_id,

    'session_id_present': bool(sample_unit.session_id),

    'conversation_count': len(sample_unit.conversation_ids),

    'turn_count': len(sample_unit.turns),

    'tool_call_count': len(sample_unit.tool_calls),

    'event_count': len(sample_trace.events),

    'canonical_utf8_bytes': len(packet_a.canonical_json.encode('utf-8')),

    'representation_truncated': packet_a.truncated,

    'content_sha256': hashlib.sha256(packet_a.canonical_json.encode('utf-8')).hexdigest(),

}

display(pd.DataFrame([sanitized]))

print('Canonical representation is deterministic; raw content remains hidden in this runbook.')

## 6. Run Random Sampling Prototype



Exercise the production `random_sampling` path with a fixed seed on a bounded interactive slice. The selector allocates an exact target through tenant, agent, turn-count-band, and channel strata, then draws without replacement.

In [ ]:
from time import perf_counter

from sampling_comparison.v2_experiment import LAST_SELECTION_MECHANISMS



demo = slice_dataset(data, limit=300)

demo_budget = 20

started = perf_counter()

random_ids = select_ids_for_method(demo, method='random_sampling_stratified', budget_pct=demo_budget, repetition_seed=SEED)

random_seconds = perf_counter() - started

random_repeat = select_ids_for_method(demo, method='random_sampling_stratified', budget_pct=demo_budget, repetition_seed=SEED)

assert random_ids == random_repeat

random_run = {

    'method': 'random_sampling_stratified',

    'owner': LAST_SELECTION_MECHANISMS['random_sampling_stratified']['owner'],

    'population': len(demo.unit_ids),

    'selected': len(random_ids),

    'budget_pct': demo_budget,

    'seed': SEED,

    'runtime_seconds': random_seconds,

    'repeatable': random_ids == random_repeat,

}

display(pd.DataFrame([random_run]))

## 7. Run MinHash LSH Sampling Prototype



Exercise the production `minhash_sampling` implementation: canonical evidence, bounded 3-token shingles, 128-value signatures, 32 bands × 4 rows, candidate-bucket union, exact signature re-ranking, and adaptive admission.

In [ ]:
started = perf_counter()

minhash_ids = select_ids_for_method(demo, method='adaptive_minhash_32x4', budget_pct=demo_budget, repetition_seed=SEED)

minhash_seconds = perf_counter() - started

minhash_repeat = select_ids_for_method(demo, method='adaptive_minhash_32x4', budget_pct=demo_budget, repetition_seed=SEED)

assert minhash_ids == minhash_repeat

minhash_mechanism = LAST_SELECTION_MECHANISMS['adaptive_minhash_32x4']

minhash_run = {

    'method': 'adaptive_minhash_32x4',

    'owner': minhash_mechanism['owner'],

    'population': len(demo.unit_ids),

    'selected': len(minhash_ids),

    'budget_cap_pct': demo_budget,

    'bands': MINHASH_BANDS,

    'rows': MINHASH_ROWS,

    'permutations': MINHASH_PERMUTATIONS,

    'runtime_seconds': minhash_seconds,

    'repeatable': minhash_ids == minhash_repeat,

}

display(pd.DataFrame([minhash_run]))

display(pd.DataFrame([minhash_mechanism]))

## 8. Compress Sessions for Embedding



Apply the retained full-session evidence compression. Measure byte reduction against a simple unbounded event serialization, verify mandatory structure survives, and avoid displaying source text. The compressed canonical packet is also the auditable payload retained for optional judging.

In [ ]:
compression_rows = []

for trace in demo.traces[:20]:

    raw_shape = json.dumps([

        {'role': event.role, 'text': event.text, 'tool_name': event.tool_name, 'arguments': event.arguments, 'output': event.output}

        for event in trace.events

    ], sort_keys=True, default=str)

    packet = packet_builder.build(trace)

    raw_bytes = len(raw_shape.encode('utf-8'))

    compressed_bytes = len(packet.canonical_json.encode('utf-8'))

    compression_rows.append({

        'trace_id': trace.trace_id,

        'raw_shape_bytes': raw_bytes,

        'compressed_bytes': compressed_bytes,

        'reduction_pct': 1.0 - compressed_bytes / max(raw_bytes, 1),

        'truncated': packet.truncated,

        'has_session_schema': 'session' in packet.canonical_json,

    })

compression_df = pd.DataFrame(compression_rows)

display(compression_df.head())

display(compression_df[['raw_shape_bytes', 'compressed_bytes', 'reduction_pct']].describe())

assert compression_df['has_session_schema'].all()

## 9. Run Full-Session Embedding Sampling Prototype



Exercise `trace_sampling.FullSessionEmbeddingPrototype`: prepare the bounded packet once, obtain a deterministic unit vector, cluster by exact same-agent cosine search in memory, and pass semantic novelty to the shared adaptive sampler.

In [ ]:
started = perf_counter()

embedding_ids = select_ids_for_method(demo, method='adaptive_embedding_fullsession', budget_pct=demo_budget, repetition_seed=SEED)

embedding_seconds = perf_counter() - started

embedding_repeat = select_ids_for_method(demo, method='adaptive_embedding_fullsession', budget_pct=demo_budget, repetition_seed=SEED)

assert embedding_ids == embedding_repeat

embedding_mechanism = LAST_SELECTION_MECHANISMS['adaptive_embedding_fullsession']

embedding_run = {

    'method': 'adaptive_embedding_fullsession',

    'owner': embedding_mechanism['owner'],

    'population': len(demo.unit_ids),

    'selected': len(embedding_ids),

    'budget_cap_pct': demo_budget,

    'profile_max_tokens': EMBEDDING_MAX_TOKENS,

    'runtime_seconds': embedding_seconds,

    'repeatable': embedding_ids == embedding_repeat,

}

display(pd.DataFrame([embedding_run]))

display(pd.DataFrame([embedding_mechanism]))



permuted_demo = with_permuted_labels(demo, seed=991)

for method, expected_ids in {

    'random_sampling_stratified': random_ids,

    'adaptive_minhash_32x4': minhash_ids,

    'adaptive_embedding_fullsession': embedding_ids,

}.items():

    actual = select_ids_for_method(permuted_demo, method=method, budget_pct=demo_budget, repetition_seed=SEED)

    assert actual == expected_ids, f'Label leakage detected for {method}'

print('All three production prototype selections are invariant to expected-label permutation.')

## 10. Evaluate Samples Against Expected Labels



Score the three demo selections only after membership is fixed. Use the V2 definitions: absolute selected-rate error versus census, fraction saved, and source-namespaced concept coverage.

In [ ]:
from sampling_comparison.v2_experiment import score_selection



demo_scores = []

for method, ids in {

    'random_sampling_stratified': random_ids,

    'adaptive_minhash_32x4': minhash_ids,

    'adaptive_embedding_fullsession': embedding_ids,

}.items():

    row = score_selection(demo, method=method, budget_pct=demo_budget, repetition=0, selected_ids=ids)

    demo_scores.append({

        'method': method,

        'selected_count': row['selected_count'],

        'selected_pass_rate': row['selected_pass_rate'],

        'census_pass_rate': row['census_pass_rate'],

        'absolute_error': row['absolute_error'],

        'fraction_saved': row['fraction_saved'],

        'concept_coverage': row['concept_coverage'],

    })

demo_score_df = pd.DataFrame(demo_scores)

display(demo_score_df)



overlap = []

sets = {'Random': set(random_ids), 'MinHash LSH': set(minhash_ids), 'Embedding': set(embedding_ids)}

for left, left_ids in sets.items():

    for right, right_ids in sets.items():

        if left < right:

            overlap.append({'left': left, 'right': right, 'intersection': len(left_ids & right_ids), 'jaccard': len(left_ids & right_ids) / max(len(left_ids | right_ids), 1)})

display(pd.DataFrame(overlap))

## 11. Compare Sampling Quality, Coverage, and Performance



Load the retained full V2 aggregates by default. `FULL_RUN=True` can create a new smoke or full production-backed bundle in a timestamped directory; the retained reference is never overwritten by this cell. Confidence intervals shown here summarize repeated deterministic replay results, not production-population uncertainty.

In [ ]:
SMOKE_CONFIG = {'budget_pcts': BUDGET_PCTS, 'outcome_repetitions': 1, 'quadrant_replays': 1, 'throughput_replays': 1}

FULL_CONFIG = {'budget_pcts': BUDGET_PCTS, 'outcome_repetitions': 3, 'quadrant_replays': 3, 'throughput_replays': 2}



if FULL_RUN:

    config = SMOKE_CONFIG if RUN_MODE == 'smoke' else FULL_CONFIG

    RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

    active_bundle = run_v2_experiment_bundle(

        data=data,

        seed=SEED,

        output_dir=RUN_OUTPUT_DIR,

        **config,

    )

    ACTIVE_DIR = RUN_OUTPUT_DIR

    print(f'Created new {RUN_MODE} bundle at {ACTIVE_DIR}')

else:

    ACTIVE_DIR = REFERENCE_DIR

    artifacts = load_v2_artifacts(default_inputs(ACTIVE_DIR))

    validate_v2_artifacts(artifacts)

    print('Loaded and validated retained V2 artifacts; no experiment rerun.')



aggregate = json.loads((ACTIVE_DIR / 'aggregate.json').read_text(encoding='utf-8'))

outcome_rows = []

diagnostics = aggregate['outcome']['aggregate_budget_diagnostics']

for method, budgets in aggregate['outcome']['aggregate_means'].items():

    for budget, metrics in budgets.items():

        diag = diagnostics[f'{method}|b{budget}']

        outcome_rows.append({

            'method': method,

            'budget_pct': int(budget),

            **metrics,

            'realized_keep_rate': diag['realized_keep_rate_mean'],

            'historical_keep_rate': diag['per_corpus']['historical_300']['mean_keep_rate'],

            'dense_keep_rate': diag['per_corpus']['dense_2500']['mean_keep_rate'],

        })

outcome_df = pd.DataFrame(outcome_rows).sort_values(['budget_pct', 'method'])

display(outcome_df)

display(outcome_df[outcome_df['budget_pct'] == 20])

## 12. Visualize V2 Experiment Results



Recreate the central diagnostic views from the retained V2 bundle: outcome curves, concept coverage, source keep rates, quadrant representation, throughput pressure, selection overlap, and compression diagnostics.

In [ ]:
method_labels = {

    'random_sampling_stratified': 'Random',

    'adaptive_minhash_32x4': 'MinHash LSH',

    'adaptive_embedding_fullsession': 'Full-session embedding',

}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for method, group in outcome_df.groupby('method'):

    label = method_labels.get(method, method)

    ordered = group.sort_values('budget_pct')

    axes[0].plot(ordered['budget_pct'], ordered['mean_absolute_error'], marker='o', label=label)

    axes[1].plot(ordered['budget_pct'], ordered['mean_fraction_saved'], marker='o', label=label)

    axes[2].plot(ordered['budget_pct'], ordered['mean_concept_coverage'], marker='o', label=label)

axes[0].set_title('Mean absolute error vs census')

axes[1].set_title('Fraction saved')

axes[2].set_title('Concept coverage')

for axis in axes:

    axis.set_xlabel('Budget %')

    axis.grid(alpha=.25)

axes[0].legend()

plt.tight_layout()

plt.show()



quadrant = json.loads((ACTIVE_DIR / 'quadrant.json').read_text(encoding='utf-8'))

throughput = json.loads((ACTIVE_DIR / 'throughput.json').read_text(encoding='utf-8'))

quadrant_df = pd.DataFrame(quadrant['aggregate_groups'].values())

display(quadrant_df[['method', 'quadrant', 'budget_pct', 'representation_mean', 'budget_utilization_mean', 'zero_selection_agent_rate_mean']])



throughput_df = pd.DataFrame(throughput['aggregate_grid'].values())

display(throughput_df.head())

budget_view = int(throughput['config']['budgets'][0])

minhash_heat = throughput_df[(throughput_df['method'] == 'adaptive_minhash_32x4') & (throughput_df['budget_pct'] == budget_view)].pivot(index='arrival_rate_per_second', columns='eval_throughput_per_second', values='representation_mean')

display(minhash_heat.style.background_gradient(cmap='Blues'))



compression_df.plot.scatter(x='raw_shape_bytes', y='compressed_bytes', title='Compressed packet bytes vs raw event shape')

plt.grid(alpha=.25)

plt.show()

## 13. Prepare Compressed Embedding Vectors for Optional LLM Judging



The production full-session path retains both a unit vector for retrieval and the exact bounded canonical packet for auditable judge submission. This notebook never sends either artifact. Numeric vectors should remain retrieval metadata; the compact evidence packet is the default judge input.

In [ ]:
from sampling_comparison.adapters import DeterministicSessionEmbedder, DeterministicTokenizer
from trace_sampling.session_embedding import EmbeddingProfile, SessionEmbeddingCache

tokenizer = DeterministicTokenizer()
embedding_profile = EmbeddingProfile(
    model_id='offline-deterministic',
    model_version='offline-deterministic-v1',
    tokenizer_id=tokenizer.name,
    tokenizer_version=tokenizer.version,
    max_input_tokens=EMBEDDING_MAX_TOKENS,
    max_representation_utf8_bytes=32768,
 )
embedding_cache = SessionEmbeddingCache(
    DeterministicSessionEmbedder(seed=REPRESENTATION_SEED),
    tokenizer,
    embedding_profile,
    packet_builder=packet_builder,
 )

prototype = FullSessionEmbeddingPrototype(embedding_cache, packet_builder=packet_builder)
prepared_sample = prototype.prepare(sample_trace)

compact_judge_payload = prototype.build_judge_payload(prepared_sample)
vector_preview_payload = prototype.build_judge_payload(prepared_sample, include_vector=True)

evidence_bytes = len(compact_judge_payload['evidence'].encode('utf-8'))
vector_dimensions = len(vector_preview_payload['embedding_vector'])

judge_preview = {
    'enabled': ENABLE_OPTIONAL_LLM_JUDGE_EXPORT,
    'trace_id': compact_judge_payload['trace_id'],
    'agent_id': compact_judge_payload['agent_id'],
    'evidence_sha256': compact_judge_payload['evidence_sha256'],
    'canonical_evidence_utf8_bytes': evidence_bytes,
    'vector_dimensions': vector_dimensions,
    'compact_payload_has_vector': 'embedding_vector' in compact_judge_payload,
    'vector_preview_has_vector': 'embedding_vector' in vector_preview_payload,
    'network_submission': False,
}
display(pd.DataFrame([judge_preview]))

if ENABLE_OPTIONAL_LLM_JUDGE_EXPORT:
    optional_export_dir = ACTIVE_DIR / 'optional_judge_inputs'
    optional_export_dir.mkdir(parents=True, exist_ok=True)
    (optional_export_dir / 'judge_payload_metadata.json').write_text(
        json.dumps(judge_preview, indent=2, sort_keys=True),
        encoding='utf-8',
    )
    print(f'Wrote local payload metadata only at {optional_export_dir}; no network submission was performed.')
else:
    print('Optional LLM judge export is disabled; payloads were not printed or written, and no network request was made.')

## 14. Reproduce and Verify the V2 HTML Report



Validate the active artifact set and optionally regenerate a report beside it. The retained reference report is never overwritten unless you deliberately change the target path. Compare headline values and structure with the reference.

In [ ]:
active_inputs = default_inputs(ACTIVE_DIR)

active_artifacts = load_v2_artifacts(active_inputs)

validate_v2_artifacts(active_artifacts)



generated_report = ACTIVE_DIR / 'agent365-sampling-v2-report.runbook.html'

if WRITE_REPORT:

    write_v2_html_report(output_path=generated_report, inputs=active_inputs)

    print(f'Generated report: {generated_report}')

else:

    print(f'Report generation disabled; retained report remains {REFERENCE_REPORT}')



reference_aggregate = json.loads((REFERENCE_DIR / 'aggregate.json').read_text(encoding='utf-8'))

active_aggregate = json.loads((ACTIVE_DIR / 'aggregate.json').read_text(encoding='utf-8'))

comparison = {

    'population_equal': active_aggregate['population_count'] == reference_aggregate['population_count'],

    'method_set_equal': set(active_aggregate['outcome']['aggregate_means']) == set(reference_aggregate['outcome']['aggregate_means']),

    'budget_set_equal': all(set(v) == {'5', '10', '20', '30', '50'} for v in active_aggregate['outcome']['aggregate_means'].values()),

    'reference_report_exists': REFERENCE_REPORT.exists(),

}

display(pd.DataFrame([comparison]))

assert all(comparison.values())

## 15. Export Reproducibility Artifacts and Execution Manifest



Validate every ExternalEvalSnapshot JSONL row and hash, verify the production storage manifest, and write a runbook execution manifest containing paths, configurations, checksums, and dependency versions. The retained reference report is not modified.

In [ ]:
snapshot_manifest = json.loads((ACTIVE_DIR / 'external_eval_snapshots' / 'manifest.json').read_text(encoding='utf-8'))

snapshot_checks = []

for method, metadata in snapshot_manifest['methods_files'].items():

    path = Path(metadata['path'])

    if not path.is_absolute():

        path = REPO_ROOT / path

    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

    for payload in rows:

        validate_external_eval_snapshot(payload)

        assert 'tenantId' not in payload

        for result in payload['results']:

            for metric in result['metrics']:

                assert metric['model'] == 'dataset-expected-label'

                assert metric['score'] in {0, 1}

                assert 'reasoning' not in metric

    actual_hash = sha256_file(path)

    snapshot_checks.append({'method': method, 'path': str(path.relative_to(REPO_ROOT)), 'rows': len(rows), 'hash_match': actual_hash == metadata['sha256']})

snapshot_check_df = pd.DataFrame(snapshot_checks)

display(snapshot_check_df)

assert snapshot_check_df['hash_match'].all() and snapshot_manifest['not_posted'] is True



storage_manifest = json.loads((ACTIVE_DIR / 'production_storage_manifest.json').read_text(encoding='utf-8'))

assert storage_manifest['scope']['implemented'] is False

assert storage_manifest['authoritative_state']['source'] == 'ESP/Cosmos'

assert storage_manifest['ppapi_contract_requirements']['route'] == 'POST /evals/service/results?api-version=1'



execution_manifest = {

    'runbook_version': 'sampling-v2-runbook-v1',

    'created_at_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),

    'full_run': FULL_RUN,

    'run_mode': RUN_MODE,

    'seed': SEED,

    'active_artifact_dir': str(ACTIVE_DIR.relative_to(REPO_ROOT)),

    'synthetic_sources': {row['corpus_id']: {'path': row['path'], 'sha256': row['actual_sha256']} for row in hash_rows},

    'prototype_packages': ['random_sampling', 'minhash_sampling', 'trace_sampling'],

    'expected_label_scoring': True,

    'llm_judge_called': False,

    'snapshots_posted': False,

    'dependency_versions': {name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'matplotlib']},

}

manifest_path = ACTIVE_DIR / 'runbook_execution_manifest.json'

manifest_path.write_text(json.dumps(execution_manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')

display(pd.DataFrame([execution_manifest]))

print(f'Execution manifest written to {manifest_path}. No LLM call, snapshot POST, or Azure mutation occurred.')